# HEAT1: does short-term heat transfer between correlated stocks?

Deep models (BiLSTM, CNN-ResNet, Transformer) read the last hour of a stock and of its 5 most
correlated peers, and predict its market-relative return over the next 30 minutes, 1, 2 and 4 hours.
Two controls, same model and budget: **no peers** (the stock's own reversal only) and **5 random
peers** (E12's control). Transfer = how much the peers cut the prediction error, in nats. The
**heat map** says which stock's moves feed which.

Data: the thesis 5-minute SIP bars on Drive (S&P 500, 2016–2026). The rule is registered in
[`configs/heat_v1.yaml`](https://github.com/Gaire-commits/Transfer-entropy-/blob/main/configs/heat_v1.yaml)
(2026-10-20). **Edge** = all of: net of 2 bps above zero on the test days (2024 onward, 99% day-block
interval), beats no peers, beats random peers, positive in both halves, survives false-discovery
control over the 12 cells.

**Runtime:** GPU (Runtime → Change runtime type → T4 or better), High-RAM if available.
Run top to bottom. Don't use "Run all" after the first time: section 4 scores the test days.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
    !pip install -q pyyaml
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO)
sys.path.insert(0, REPO)
for name in [m for m in sys.modules if m == "heat" or m.startswith("heat.")]:
    del sys.modules[name]  # re-running this cell after a pull uses the new code

import torch
from IPython.display import Markdown, display

DATA = os.environ.get("HEAT_DATA_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/data/bars_sip")
OUT = os.environ.get("HEAT_OUT_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (deep models will be slow)")
print("SIP bars found:", os.path.isdir(os.path.join(DATA, "5Min")))
print("outputs:", OUT)

## 2. Controls (about 2 minutes)

Before real data: on synthetic stocks with a planted leader, the experiment must find the edge and
name the leader in the heat map; with nothing planted it must find nothing. Plus the no-look-ahead
checks and the frozen-config check.

In [ ]:
!{sys.executable} -m pytest -q tests -W ignore

## 3. Check run on the real data (inside the training years)

Builds the panel once (every stock's market-relative returns on the thesis grid; a few minutes,
saved to the outputs folder), then a short run that trains to 2019, validates on 2020 and scores
**2021**. It never touches the registered test days, so it's safe to run as often as needed. It shows
whether everything works and gives a feel for the numbers; it is not the result.

In [ ]:
!{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --check
display(Markdown(open(os.path.join(OUT, "check", "report.md")).read()))

## 4. The registered run: scores the test days

One architecture per cell, so each fits in a Colab session. Each is saved to Drive as soon as it's
scored, and a scored one is skipped if you run its cell again, so a dropped session just resumes. The
verdict appears once all three are in.

In [ ]:
!{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --archs bilstm

In [ ]:
!{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --archs resnet1d

In [ ]:
!{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --archs transformer

## 5. Result

In [ ]:
display(Markdown(open(os.path.join(OUT, "report.md")).read()))

### Where the heat goes

For each source → target pair (the source is one of the target's correlated peers), the share of
the target's prediction error that the source's last hour removes. Large, stable shares across
architectures are the interesting links.

In [ ]:
import pandas as pd
heat = pd.read_csv(os.path.join(OUT, "heat_flows.csv"))
stable = (heat.groupby(["hold", "source", "target"])["heat_share"]
              .agg(["mean", "min", "count"]).query("count == 3 and min > 0")
              .sort_values("mean", ascending=False))
print("pairs positive in all three architectures, by hold:")
stable.groupby(level="hold").head(10)